# LeetCode #75 — Sort Colors: Dutch National Flag, ba con trỏ, một lượt

**Mức độ:** Medium · **Chủ đề:** Array, Two Pointers, Sorting

Skeleton trên LeetCode:

```python
class Solution:
    def sortColors(self, nums: List[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
```

Mục tiêu của notebook:

1. Sắp xếp mảng chỉ có 3 giá trị **không dùng hàm sort** có sẵn.
2. Cách 1: **đếm** (counting sort) — hai lượt.
3. Cách 2: **Dutch National Flag** (Dijkstra) — ba con trỏ `low`, `mid`, `high`, **một lượt**.
4. Hiểu bất biến của bốn vùng và vì sao **không tăng `mid`** khi đổi với `high`.

# 1. Bài toán là gì?

Mảng `nums` gồm `n` phần tử, mỗi phần tử là `0` (đỏ), `1` (trắng), hoặc `2` (xanh). Sắp xếp **tại chỗ** sao cho các màu giống nhau đứng liền nhau, theo thứ tự đỏ, trắng, xanh. **Không được** dùng hàm sort của thư viện.

**Constraints (theo đề):**

- `n == nums.length`, `1 <= n <= 300`
- `nums[i]` là `0`, `1`, hoặc `2`.

**Follow up (theo đề):** bạn có nghĩ ra thuật toán **một lượt** với bộ nhớ hằng không?

---

## Ví dụ

```text
[2, 0, 2, 1, 1, 0]  ->  [0, 0, 1, 1, 2, 2]
[2, 0, 1]           ->  [0, 1, 2]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Hai con trỏ (two pointers)
2. Sắp xếp (sorting)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [1]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [2]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.2 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [3]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [4]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [5]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.3 Áp dụng vào bài này

Có thể **đếm** rồi ghi lại (counting sort), nhưng lời giải một lượt là **ba con trỏ** (Dutch National Flag): `low` (vùng 0), `mid` (đang xét), `high` (vùng 2) — mở rộng của kỹ thuật con trỏ đọc/ghi.

# 3. Cách 1: Đếm rồi ghi lại (hai lượt)

Lượt 1 đếm số `0`, `1`, `2`; lượt 2 ghi đè mảng.

In [6]:
from typing import List

def sort_colors_count(nums: List[int]) -> None:
    count = [0, 0, 0]
    for x in nums:
        count[x] += 1
    i = 0
    for color in range(3):
        for _ in range(count[color]):
            nums[i] = color
            i += 1

a = [2, 0, 2, 1, 1, 0]
sort_colors_count(a)
print(a)

[0, 0, 1, 1, 2, 2]


`O(n)` thời gian, `O(1)` bộ nhớ (mảng đếm cố định 3 ô) — đã rất tốt. Nhưng follow-up muốn **một lượt**. Ngoài ra, cách đếm chỉ dùng được khi phần tử chính là "nhãn"; nếu cần sắp xếp các **object** theo 3 loại (giữ nguyên object), ta phải **di chuyển** chúng, không ghi đè được.

# 4. Ý tưởng chính: chia mảng thành bốn vùng

Duy trì ba con trỏ `low`, `mid`, `high`:

```text
[ 0 0 0 | 1 1 1 | ? ? ? ? | 2 2 2 ]
  0..low-1  low..mid-1  mid..high  high+1..n-1
     = 0        = 1     chưa xét      = 2
```

Xét `nums[mid]`:

- `0` → đổi với `nums[low]`, rồi `low += 1`, `mid += 1`.
- `1` → đã đúng chỗ, `mid += 1`.
- `2` → đổi với `nums[high]`, rồi `high -= 1` — **không** tăng `mid`, vì phần tử vừa đổi về từ vùng "chưa xét" chưa được kiểm tra.

Dừng khi `mid > high` (vùng chưa xét rỗng).

In [7]:
def sort_colors_dnf(nums: List[int]) -> None:
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1
            mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1

a = [2, 0, 2, 1, 1, 0]
sort_colors_dnf(a)
print(a)

[0, 0, 1, 1, 2, 2]


**Tại sao khi gặp `0` lại được tăng `mid`?** Phần tử ở `low` mà ta đổi sang vị trí `mid` thuộc vùng `[low, mid-1]` = toàn `1` (hoặc `low == mid`, khi đó đổi với chính nó). Nên sau khi đổi, `nums[mid]` là `1` — đã xét xong.

# 5. Mô phỏng

In [8]:
def trace_dnf(nums: List[int]) -> None:
    low, mid, high = 0, 0, len(nums) - 1
    print(f"{'nums':<22} low mid high  hành động")
    while mid <= high:
        state = f"{str(nums):<22} {low:>3} {mid:>3} {high:>4}"
        x = nums[mid]
        if x == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1; mid += 1
            act = "0: đổi với low, low++, mid++"
        elif x == 1:
            mid += 1
            act = "1: mid++"
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1
            act = "2: đổi với high, high--"
        print(f"{state}  {act}")
    print(f"{str(nums):<22} {low:>3} {mid:>3} {high:>4}  mid > high -> dừng")

trace_dnf([2, 0, 2, 1, 1, 0])

nums                   low mid high  hành động
[2, 0, 2, 1, 1, 0]       0   0    5  2: đổi với high, high--
[0, 0, 2, 1, 1, 2]       0   0    4  0: đổi với low, low++, mid++
[0, 0, 2, 1, 1, 2]       1   1    4  0: đổi với low, low++, mid++
[0, 0, 2, 1, 1, 2]       2   2    4  2: đổi với high, high--
[0, 0, 1, 1, 2, 2]       2   2    3  1: mid++
[0, 0, 1, 1, 2, 2]       2   3    3  1: mid++
[0, 0, 1, 1, 2, 2]       2   4    3  mid > high -> dừng


Mỗi bước, hoặc `mid` tăng, hoặc `high` giảm → vùng chưa xét `[mid, high]` co lại 1 → tối đa `n` bước.

# 6. Phiên bản đúng format LeetCode

In [9]:
class Solution:
    def sortColors(self, nums: List[int]) -> None:
        low, mid, high = 0, 0, len(nums) - 1
        while mid <= high:
            if nums[mid] == 0:
                nums[low], nums[mid] = nums[mid], nums[low]
                low += 1
                mid += 1
            elif nums[mid] == 1:
                mid += 1
            else:
                nums[mid], nums[high] = nums[high], nums[mid]
                high -= 1

In [10]:
import random

def run(f, nums):
    nums = nums[:]
    assert f(nums) is None
    return nums

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", run(sol.sortColors, [2, 0, 2, 1, 1, 0]), [0, 0, 1, 1, 2, 2])
check("Ví dụ 2", run(sol.sortColors, [2, 0, 1]), [0, 1, 2])
check("Một phần tử", run(sol.sortColors, [1]), [1])
check("Toàn 2", run(sol.sortColors, [2, 2, 2]), [2, 2, 2])
check("Đã sắp xếp", run(sol.sortColors, [0, 1, 2]), [0, 1, 2])
check("Ngược", run(sol.sortColors, [2, 2, 1, 1, 0, 0]), [0, 0, 1, 1, 2, 2])

random.seed(75)
for _ in range(5000):
    nums = [random.randint(0, 2) for _ in range(random.randint(1, 15))]
    expected = sorted(nums)
    assert run(sol.sortColors, nums) == expected == run(sort_colors_count, nums)
print("5000 mảng ngẫu nhiên: khớp sorted()")

Ví dụ 1              PASS  [0, 0, 1, 1, 2, 2]
Ví dụ 2              PASS  [0, 1, 2]
Một phần tử          PASS  [1]
Toàn 2               PASS  [2, 2, 2]
Đã sắp xếp           PASS  [0, 1, 2]
Ngược                PASS  [0, 0, 1, 1, 2, 2]
5000 mảng ngẫu nhiên: khớp sorted()


# 7. Vì sao đúng? Độ phức tạp?

**Bất biến** (đúng trước mỗi vòng lặp):

- `nums[0 .. low-1]` toàn `0`
- `nums[low .. mid-1]` toàn `1`
- `nums[high+1 .. n-1]` toàn `2`
- `nums[mid .. high]` chưa xét

Ban đầu `low = mid = 0`, `high = n-1`: ba vùng đầu rỗng, đúng. Mỗi nhánh của vòng lặp giữ bất biến (đã giải thích ở mục 4). Khi `mid > high`, vùng chưa xét rỗng → mảng gồm `0…0 1…1 2…2`.

```text
Time:  O(n), một lượt — mỗi vòng lặp thu hẹp vùng chưa xét [mid, high] đi 1
Space: O(1)
```

Số phép đổi chỗ tối đa `n`.

# 8. Những lỗi dễ mắc

## Lỗi 1: Tăng `mid` sau khi đổi với `high`

In [11]:
def wrong_inc_mid(nums: List[int]) -> None:
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1; mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1
            mid += 1                              # SAI: phần tử mới ở mid chưa được xét

print("Sai :", run(wrong_inc_mid, [1, 2, 0]), "| Đúng:", run(sol.sortColors, [1, 2, 0]))
print("(Với [2, 0, 2, 1, 1, 0] bản sai tình cờ vẫn đúng — test ít là không đủ!)", run(wrong_inc_mid, [2, 0, 2, 1, 1, 0]))

Sai : [1, 0, 2] | Đúng: [0, 1, 2]
(Với [2, 0, 2, 1, 1, 0] bản sai tình cờ vẫn đúng — test ít là không đủ!) [0, 0, 1, 1, 2, 2]


## Lỗi 2: Điều kiện `while mid < high`

Khi `mid == high` vẫn còn **một** phần tử chưa xét.

In [12]:
def wrong_strict(nums: List[int]) -> None:
    low, mid, high = 0, 0, len(nums) - 1
    while mid < high:                             # SAI
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1; mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1

print("Sai :", run(wrong_strict, [1, 0]), "| Đúng:", run(sol.sortColors, [1, 0]))

Sai : [1, 0] | Đúng: [0, 1]


## Lỗi 3: Hai lượt "phân hoạch" nhưng lượt 2 quét lại từ đầu

Cách hai lượt hợp lệ: lượt 1 đưa mọi `0` lên đầu (như partition), lượt 2 đưa mọi `1` ngay sau vùng `0` — lượt 2 phải bắt đầu **sau** vùng `0`. Không sai nếu quét lại từ đầu, chỉ thừa.

## Lỗi 4: Dùng `nums.sort()` hoặc `sorted(nums)`

Đề cấm. Ngoài ra `nums = sorted(nums)` còn không sửa được list bên ngoài hàm (chỉ gán lại biến cục bộ).

In [13]:
def wrong_rebind(nums: List[int]) -> None:
    nums = sorted(nums)                           # SAI: chỉ gán lại tên cục bộ

a = [2, 0, 1]
wrong_rebind(a)
print("Sau wrong_rebind:", a)

Sau wrong_rebind: [2, 0, 1]


# 9. Mẹo nhận diện pattern

**Phân hoạch (partition) với con trỏ** khi cần chia mảng thành vài nhóm **tại chỗ**:

- 2 nhóm → hai con trỏ (như partition của Quicksort, #905 Sort Array By Parity, #283 Move Zeroes).
- 3 nhóm → Dutch National Flag (bài này; Quicksort 3-way để xử lý nhiều phần tử trùng pivot).
- `k` nhóm với `k` nhỏ → đếm (counting sort).

Bài liên quan: **#283 Move Zeroes**, **#905 Sort Array By Parity**, **#2161 Partition Array According to Given Pivot**, **#324 Wiggle Sort II**.

## Template DNF với pivot

```python
low, mid, high = 0, 0, n - 1
while mid <= high:
    if a[mid] < pivot:  swap(low, mid); low += 1; mid += 1
    elif a[mid] > pivot: swap(mid, high); high -= 1
    else: mid += 1
```

# 10. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chạy DNF trên `[1, 2, 0, 0, 2, 1]`, ghi lại `low, mid, high` mỗi bước.

## Bài 2 — Điền chỗ trống

In [14]:
def exercise_2_sort_colors(nums: List[int]) -> None:
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        # TODO: ba trường hợp 0 / 1 / 2
        pass


# a = [2, 0, 2, 1, 1, 0]; exercise_2_sort_colors(a); assert a == [0, 0, 1, 1, 2, 2]
# a = [2, 0, 1]; exercise_2_sort_colors(a); assert a == [0, 1, 2]

## Bài 3 — Biến thể: phân hoạch quanh pivot (3-way partition)

Sắp lại tại chỗ sao cho các phần tử `< pivot` đứng trước, `== pivot` ở giữa, `> pivot` ở sau (thứ tự trong mỗi nhóm tùy ý). Trả về cặp `(lt, gt)`: `nums[lt..gt]` là vùng bằng `pivot`.

```python
a = [5, 1, 9, 5, 3, 7, 5]
three_way(a, 5)  -> a có dạng [<5, <5, 5, 5, 5, >5, >5], trả (2, 4)
```

In [15]:
def exercise_3_three_way(nums: List[int], pivot: int):
    # TODO
    pass


# a = [5, 1, 9, 5, 3, 7, 5]; lt, gt = exercise_3_three_way(a, 5); assert (lt, gt) == (2, 4) and all(x < 5 for x in a[:2]) and a[2:5] == [5, 5, 5] and all(x > 5 for x in a[5:])
# a = [1, 2, 3]; lt, gt = exercise_3_three_way(a, 10); assert (lt, gt) == (3, 2) and sorted(a) == [1, 2, 3]

## Bài 4 — Biến thể: Move Zeroes (LeetCode #283)

Đưa mọi số `0` về **cuối**, giữ **thứ tự tương đối** các số khác 0. Tại chỗ.

```python
move_zeroes([0, 1, 0, 3, 12]) -> [1, 3, 12, 0, 0]
```

Gợi ý nhanh: DNF **không** giữ thứ tự. Dùng con trỏ `write`: chép các số khác 0 lên trước theo thứ tự (hoặc đổi chỗ).

In [16]:
def exercise_4_move_zeroes(nums: List[int]) -> None:
    # TODO
    pass


# a = [0, 1, 0, 3, 12]; exercise_4_move_zeroes(a); assert a == [1, 3, 12, 0, 0]
# a = [0]; exercise_4_move_zeroes(a); assert a == [0]

## Bài 5 — Mini challenge: Sort Colors II — `k` màu

Mảng gồm các màu `1..k` (`k <= n`). Sắp xếp tại chỗ, `O(1)` bộ nhớ phụ (không dùng mảng đếm độ dài `k`), thời gian `O(n log k)`.

```python
sort_colors_k([3, 2, 2, 1, 4], 4) -> [1, 2, 2, 3, 4]
```

Gợi ý nhanh: **chia để trị trên màu**: phân hoạch đoạn `[l, r]` theo `mid = (lo_color + hi_color) // 2` (phần `<= mid` sang trái), rồi đệ quy hai nửa với khoảng màu tương ứng. Độ sâu đệ quy `O(log k)`.

In [17]:
def exercise_5_sort_colors_k(nums: List[int], k: int) -> None:
    # TODO
    pass


# a = [3, 2, 2, 1, 4]; exercise_5_sort_colors_k(a, 4); assert a == [1, 2, 2, 3, 4]
# a = [2, 1, 1, 2, 2]; exercise_5_sort_colors_k(a, 2); assert a == [1, 1, 2, 2, 2]

# 11. Gợi ý

**Bài 1**
- Mức 1: bước đầu `nums[0] = 1` → `mid++`.

**Bài 3**
- Mức 1: giống hệt DNF, thay `== 0` bằng `< pivot`, `== 2` bằng `> pivot`.
- Mức 2: khi dừng, vùng bằng `pivot` là `[low, high]` → trả `(low, high)`.

**Bài 4**
- Mức 1: `write = 0`; với mỗi `i`, nếu `nums[i] != 0` thì đổi `nums[write], nums[i]` và `write += 1`.

**Bài 5**
- Mức 1: hàm `solve(l, r, clo, chi)`: nếu `clo >= chi` hoặc `l >= r` thì dừng.
- Mức 2: `cmid = (clo + chi) // 2`; partition hai nhóm `<= cmid` / `> cmid` trên `[l, r]`, được vị trí chia `p`.
- Mức 3: `solve(l, p - 1, clo, cmid)` và `solve(p, r, cmid + 1, chi)`.

# 12. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [18]:
trace_dnf([1, 2, 0, 0, 2, 1])

nums                   low mid high  hành động
[1, 2, 0, 0, 2, 1]       0   0    5  1: mid++
[1, 2, 0, 0, 2, 1]       0   1    5  2: đổi với high, high--
[1, 1, 0, 0, 2, 2]       0   1    4  1: mid++
[1, 1, 0, 0, 2, 2]       0   2    4  0: đổi với low, low++, mid++
[0, 1, 1, 0, 2, 2]       1   3    4  0: đổi với low, low++, mid++
[0, 0, 1, 1, 2, 2]       2   4    4  2: đổi với high, high--
[0, 0, 1, 1, 2, 2]       2   4    3  mid > high -> dừng


## Đáp án Bài 2

In [19]:
def answer_2_sort_colors(nums: List[int]) -> None:
    sort_colors_dnf(nums)

a = [2, 0, 2, 1, 1, 0]; answer_2_sort_colors(a); assert a == [0, 0, 1, 1, 2, 2]
a = [2, 0, 1]; answer_2_sort_colors(a); assert a == [0, 1, 2]
print("Bài 2 PASS (giống sort_colors_dnf ở mục 4)")

Bài 2 PASS (giống sort_colors_dnf ở mục 4)


## Đáp án Bài 3

In [20]:
def answer_3_three_way(nums: List[int], pivot: int):
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if nums[mid] < pivot:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1; mid += 1
        elif nums[mid] > pivot:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1
        else:
            mid += 1
    return low, high

a = [5, 1, 9, 5, 3, 7, 5]; lt, gt = answer_3_three_way(a, 5); assert (lt, gt) == (2, 4) and all(x < 5 for x in a[:2]) and a[2:5] == [5, 5, 5] and all(x > 5 for x in a[5:])
a = [1, 2, 3]; lt, gt = answer_3_three_way(a, 10); assert (lt, gt) == (3, 2) and sorted(a) == [1, 2, 3]
print("Bài 3 PASS")

Bài 3 PASS


Đây chính là bước phân hoạch của **Quicksort 3-way**: các phần tử bằng `pivot` đã ở đúng chỗ, chỉ cần đệ quy hai vùng `< pivot` và `> pivot`. Nhờ vậy mảng nhiều phần tử trùng không làm Quicksort suy biến thành `O(n²)`.

## Đáp án Bài 4

In [21]:
def answer_4_move_zeroes(nums: List[int]) -> None:
    write = 0
    for i in range(len(nums)):
        if nums[i] != 0:
            nums[write], nums[i] = nums[i], nums[write]
            write += 1

a = [0, 1, 0, 3, 12]; answer_4_move_zeroes(a); assert a == [1, 3, 12, 0, 0]
a = [0]; answer_4_move_zeroes(a); assert a == [0]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [22]:
def answer_5_sort_colors_k(nums: List[int], k: int) -> None:
    def solve(l, r, clo, chi):
        if l >= r or clo >= chi:
            return
        cmid = (clo + chi) // 2
        p = l
        for i in range(l, r + 1):
            if nums[i] <= cmid:
                nums[p], nums[i] = nums[i], nums[p]
                p += 1
        solve(l, p - 1, clo, cmid)
        solve(p, r, cmid + 1, chi)
    solve(0, len(nums) - 1, 1, k)

a = [3, 2, 2, 1, 4]; answer_5_sort_colors_k(a, 4); assert a == [1, 2, 2, 3, 4]
a = [2, 1, 1, 2, 2]; answer_5_sort_colors_k(a, 2); assert a == [1, 1, 2, 2, 2]
for _ in range(2000):
    k = random.randint(1, 8)
    nums = [random.randint(1, k) for _ in range(random.randint(1, 20))]
    b = nums[:]
    answer_5_sort_colors_k(b, k)
    assert b == sorted(nums)
print("Bài 5 PASS (2000 mảng ngẫu nhiên)")

Bài 5 PASS (2000 mảng ngẫu nhiên)


# 13. Tổng kết

> Ba con trỏ chia mảng thành bốn vùng `0 | 1 | ? | 2`. Gặp `0`: đổi với `low`, tăng cả `low` và `mid`. Gặp `1`: tăng `mid`. Gặp `2`: đổi với `high`, giảm `high`, **không** tăng `mid`. Một lượt, `O(1)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] `while mid <= high` (có dấu bằng)?
- [ ] Không tăng `mid` sau khi đổi với `high`?
- [ ] Sửa tại chỗ, không gán lại `nums = ...`?
- [ ] Không dùng hàm sort của thư viện?